# Parte 2 — Preprocesamiento y Feature Engineering

En esta sección partimos del dataset limpio de la Parte 1 y preparamos los datos
para el modelado:

1. División train/test
2. Limpieza: tipos de dato, duplicados y missings
3. Feature engineering: nuevas variables derivadas
4. Encoding de variables categóricas
5. Escalado de variables numéricas
6. Tabla resumen de variables del dataset

In [ ]:
#Importamos las librerías necesarias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


In [11]:
# Cargamos el CSV original
df = pd.read_csv('../data_sample/global_natural_disasters_2000_2025.csv')

# Replicamos exactamente lo que hizo la Parte 1

df['es_severo'] = (df['magnitude'] >= 6.0).astype(int)
df = df.drop(columns=['magnitude', 'country', 'deaths'])

print(f"Shape del dataset: {df.shape}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())
df.head()

Shape del dataset: (46419, 8)

Columnas disponibles:
['disaster_type', 'date', 'latitude', 'longitude', 'depth_km', 'location', 'source', 'es_severo']


,disaster_type,date,latitude,longitude,depth_km,location,source,es_severo
0,Earthquake,2000-01-01,36.874,69.947,54.3,"29 km SSE of Rust?q, Afghanistan",USGS,0
1,Earthquake,2000-01-01,-60.722,153.670,10.0,west of Macquarie Island,USGS,1
2,Earthquake,2000-01-01,23.112,143.644,33.0,"Volcano Islands, Japan region",USGS,0
3,Earthquake,2000-01-02,27.559,92.498,33.0,"33 km NNE of Bomdila, India",USGS,0
4,Earthquake,2000-01-02,-17.943,-178.476,582.3,"234 km E of Levuka, Fiji",USGS,0


## 1. División Train/Test


Antes de cualquier transformación dividimos el dataset en train y test.

- **Test size:** 20%
- **Stratify:** usamos `es_severo` para mantener la proporción de clases
  (91.7% / 8.3%) en ambos conjuntos

In [12]:
# Separamos features y target

X = df.drop(columns=['es_severo'])
y = df['es_severo']

# División estratificada

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} registros ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} registros ({X_test.shape[0]/len(df)*100:.1f}%)")

# Verificamos que el stratify ha funcionado correctamente

print(f"\nProporción de clases en train:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nProporción de clases en test:")
print(y_test.value_counts(normalize=True).round(3))

Train: 37135 registros (80.0%)
Test:  9284 registros (20.0%)

Proporción de clases en train:
es_severo
0    0.917
1    0.083
Name: proportion, dtype: float64

Proporción de clases en test:
es_severo
0    0.917
1    0.083
Name: proportion, dtype: float64


## 2. Limpieza de Datos

Revisamos tipos de dato, duplicados y valores missing **solo sobre train**
para no contaminar el test set.

In [13]:
# Tipos de dato

print("Tipos de dato:")
print(X_train.dtypes)

# Duplicados

duplicados = X_train.duplicated().sum()
print(f"\nDuplicados en train: {duplicados}")

# Missings

print("\nValores missing en train:")
missings = X_train.isnull().sum()
pct_missing = (X_train.isnull().sum() / len(X_train) * 100).round(2)
resumen_missing = pd.DataFrame({
    'Missing': missings,
    'Porcentaje': pct_missing
})
print(resumen_missing[resumen_missing['Missing'] > 0])

Tipos de dato:
disaster_type     object
date              object
latitude         float64
longitude        float64
depth_km         float64
location          object
source            object
dtype: object

Duplicados en train: 1

Valores missing en train:
Empty DataFrame
Columns: [Missing, Porcentaje]
Index: []


In [14]:
#Eliminamos el duplicado que tenemos

X_train = X_train.drop_duplicates()
y_train = y_train[X_train.index]

print(f"Shape train tras eliminar duplicados: {X_train.shape}")

Shape train tras eliminar duplicados: (37134, 7)


## 3. Feature Engineering

Creamos nuevas variables que pueden mejorar la capacidad predictiva del modelo:

- **Mes** y **año**: extraídos de la columna `date` — permiten capturar patrones temporales
- **Cinturón de Fuego**: variable binaria que indica si el terremoto ocurrió en el
  Cinturón de Fuego del Pacífico, zona con mayor actividad sísmica significativa
- **Prof_grupo**: categorización de la profundidad en grupos geofísicos estándar
  (superficial, intermedio, profundo)

In [ ]:
def aplicar_feature_engineering(X):
    X = X.copy()
    
    # Extraemos mes y año de la fecha
    
    X['date'] = pd.to_datetime(X['date'])
    X['mes'] = X['date'].dt.month
    X['año'] = X['date'].dt.year
    
    # Cinturón de Fuego del Pacífico
    # Zona aproximada: longitud entre -180 y -60 (América) o entre 100 y 180 (Asia/Oceanía)
    # y latitud entre -60 y 70
    
    X['cinturon_fuego'] = (
        (
            ((X['longitude'] >= 100) & (X['longitude'] <= 180)) |
            ((X['longitude'] >= -180) & (X['longitude'] <= -60))
        ) &
        (X['latitude'] >= -60) & (X['latitude'] <= 70)
    ).astype(int)
    
    # Grupos de profundidad
    
    X['prof_grupo'] = pd.cut(
        X['depth_km'],
        bins=[-np.inf, 70, 300, np.inf],
        labels=['superficial', 'intermedio', 'profundo']
    )
    
    return X

# Aplicamos sobre train y test

X_train = aplicar_feature_engineering(X_train)
X_test  = aplicar_feature_engineering(X_test)

print("Nuevas features creadas:")
print(X_train[['mes', 'año', 'cinturon_fuego', 'prof_grupo']].head())
print(f"\nDistribución cinturon_fuego en train:")
print(X_train['cinturon_fuego'].value_counts())
print(f"\nDistribución prof_grupo en train:")
print(X_train['prof_grupo'].value_counts())

Nuevas features creadas:
       mes   año  cinturon_fuego   prof_grupo
27062   11  2014               0  superficial
1266    11  2000               1  superficial
22005   11  2011               1  superficial
27518    2  2015               1  superficial
40816   11  2022               1  superficial

Distribución cinturon_fuego en train:
cinturon_fuego
1    28793
0     8341
Name: count, dtype: int64

Distribución prof_grupo en train:
prof_grupo
superficial    30926
intermedio      4838
profundo        1370
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Eliminamos columnas que no vamos a necesitar

cols_eliminar = ['date', 'location', 'source', 'disaster_type']
X_train = X_train.drop(columns=cols_eliminar)
X_test  = X_test.drop(columns=cols_eliminar)

# Encoding ordinal de prof_grupo

ordinal_enc = OrdinalEncoder(categories=[['superficial', 'intermedio', 'profundo']])
X_train['prof_grupo'] = ordinal_enc.fit_transform(X_train[['prof_grupo']])
X_test['prof_grupo']  = ordinal_enc.transform(X_test[['prof_grupo']])

# Escalado de variables numéricas

cols_escalar = ['latitude', 'longitude', 'depth_km', 'mes', 'año']
scaler = StandardScaler()
X_train[cols_escalar] = scaler.fit_transform(X_train[cols_escalar])
X_test[cols_escalar]  = scaler.transform(X_test[cols_escalar])

print("Columnas finales del dataset:")
print(X_train.columns.tolist())
print(f"\nShape train: {X_train.shape}")
print(f"Shape test:  {X_test.shape}")
print(f"\nPrimeras filas de X_train:")
X_train.head()

Columnas finales del dataset:
['latitude', 'longitude', 'depth_km', 'mes', 'año', 'cinturon_fuego', 'prof_grupo']

Shape train: (37134, 7)
Shape test:  (9284, 7)

Primeras filas de X_train:


,latitude,longitude,depth_km,mes,año,cinturon_fuego,prof_grupo
27062,0.271712,0.451745,-0.441241,1.287730,0.176815,0,0.0
1266,-0.192734,0.540315,-0.250326,1.287730,-1.749461,1,0.0
22005,-0.742767,-1.743239,-0.240485,1.287730,-0.235958,1,0.0
27518,-0.508023,-1.772306,-0.195217,-1.283315,0.314406,1,0.0
40816,-0.301739,0.982726,-0.441241,1.287730,1.277544,1,0.0


## 5. Tabla Resumen de Variables

Resumen de todas las variables del dataset final con su tipo, origen y descripción.

In [18]:
tabla_variables = pd.DataFrame({
    'Variable': ['latitude', 'longitude', 'depth_km', 'mes', 'año', 
                 'cinturon_fuego', 'prof_grupo', 'es_severo'],
    'Tipo': ['Numérica continua', 'Numérica continua', 'Numérica continua', 
             'Numérica discreta', 'Numérica discreta',
             'Binaria', 'Categórica ordinal', 'Binaria (target)'],
    'Origen': ['Original', 'Original', 'Original', 'Engineered', 'Engineered',
               'Engineered', 'Engineered', 'Engineered (Parte 1)'],
    'Descripción': [
        'Latitud del epicentro',
        'Longitud del epicentro',
        'Profundidad del terremoto en km',
        'Mes del terremoto (1-12)',
        'Año del terremoto (2000-2025)',
        '1 si ocurrió en el Cinturón de Fuego del Pacífico',
        'Profundidad categorizada: superficial / intermedio / profundo',
        '1 = terremoto significativo (magnitud ≥ 6.0)'
    ]
})

print(tabla_variables.to_string(index=False))

      Variable               Tipo               Origen                                                   Descripción
      latitude  Numérica continua             Original                                         Latitud del epicentro
     longitude  Numérica continua             Original                                        Longitud del epicentro
      depth_km  Numérica continua             Original                               Profundidad del terremoto en km
           mes  Numérica discreta           Engineered                                      Mes del terremoto (1-12)
           año  Numérica discreta           Engineered                                 Año del terremoto (2000-2025)
cinturon_fuego            Binaria           Engineered             1 si ocurrió en el Cinturón de Fuego del Pacífico
    prof_grupo Categórica ordinal           Engineered Profundidad categorizada: superficial / intermedio / profundo
     es_severo   Binaria (target) Engineered (Parte 1)          